In [1]:
import pandas as pd
dtype_dict = {'bathrooms':float, 'waterfront':int, 'sqft_above':int, 'sqft_living15':float, 'grade':int, 'yr_renovated':int, 'price':float, 'bedrooms':float, 'zipcode':str, 'long':float, 'sqft_lot15':float, 'sqft_living':float, 'floors':str, 'condition':int, 'lat':float, 'date':str, 'sqft_basement':int, 'yr_built':int, 'id':str, 'sqft_lot':int, 'view':int}
df = pd.read_csv('kc_house_train_data.csv',dtype = dtype_dict)

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17384 entries, 0 to 17383
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             17384 non-null  object 
 1   date           17384 non-null  object 
 2   price          17384 non-null  float64
 3   bedrooms       17384 non-null  float64
 4   bathrooms      17384 non-null  float64
 5   sqft_living    17384 non-null  float64
 6   sqft_lot       17384 non-null  int32  
 7   floors         17384 non-null  object 
 8   waterfront     17384 non-null  int32  
 9   view           17384 non-null  int32  
 10  condition      17384 non-null  int32  
 11  grade          17384 non-null  int32  
 12  sqft_above     17384 non-null  int32  
 13  sqft_basement  17384 non-null  int32  
 14  yr_built       17384 non-null  int32  
 15  yr_renovated   17384 non-null  int32  
 16  zipcode        17384 non-null  object 
 17  lat            17384 non-null  float64
 18  long  

In [3]:
sqft = df['sqft_living'].tolist()
price = df['price'].tolist()

In [4]:
from modules.mylinalg import flatten
from modules.mystats import mean

def my_simple_linear_regression(x: [list], y: [list]) -> tuple[float, float]:

    x = flatten(x)
    y = flatten(y)
    
    sum_xy = sum([x[i]*y[i] for i in range(len(x))])
    sum_x = sum(x)
    sum_y = sum(y)
    sum_x2 = sum([x[i]**2 for i in range(len(x))])

    b1 = (len(x)*sum_xy - (sum_x * sum_y)) / (len(y)*sum_x2 - sum_x**2)
    b0 = mean(y) - b1*mean(x)

    return b1, b0

In [5]:
slope, intercept = my_simple_linear_regression(sqft,price)

In [6]:
def get_regression_predictions(input_feature, intercept, slope):
    predicted_values = input_feature*slope + intercept
    return predicted_values

In [7]:
get_regression_predictions(2650, intercept, slope)

700074.8459475136

In [8]:
RSS = sum([(get_regression_predictions(sqft[i], intercept, slope)-price[i])**2 for i in range(len(sqft))])
RSS

1201918354177283.0

In [9]:
def inverse_regression_predictions(output, intercept, slope):
    # solve output = intercept + slope*input_feature for input_feature. Use this equation to compute the inverse predictions:
    estimated_feature = (output - intercept)/slope
    return estimated_feature

In [10]:
inverse_regression_predictions(800000, intercept, slope)

3004.396245152277

In [11]:
inverse_regression_predictions(800000, intercept, slope)

3004.396245152277

In [12]:
RSS = sum([(get_regression_predictions(price[i], intercept, slope)-sqft[i])**2 for i in range(len(sqft))])
RSS

5.906710444060591e+20

In [13]:
from sklearn.linear_model import LinearRegression

test_df = pd.read_csv('kc_house_test_data.csv',dtype = dtype_dict)
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4229 entries, 0 to 4228
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             4229 non-null   object 
 1   date           4229 non-null   object 
 2   price          4229 non-null   float64
 3   bedrooms       4229 non-null   float64
 4   bathrooms      4229 non-null   float64
 5   sqft_living    4229 non-null   float64
 6   sqft_lot       4229 non-null   int32  
 7   floors         4229 non-null   object 
 8   waterfront     4229 non-null   int32  
 9   view           4229 non-null   int32  
 10  condition      4229 non-null   int32  
 11  grade          4229 non-null   int32  
 12  sqft_above     4229 non-null   int32  
 13  sqft_basement  4229 non-null   int32  
 14  yr_built       4229 non-null   int32  
 15  yr_renovated   4229 non-null   int32  
 16  zipcode        4229 non-null   object 
 17  lat            4229 non-null   float64
 18  long    

In [14]:
import numpy as np

bedroom_squared = test_df['bedrooms'] ** 2
bed_bath_rooms = test_df['bedrooms'] * test_df['bathrooms']
lof_sqft_living = np.log(test_df['sqft_living'])
lat_plus_long = test_df['lat'] + test_df['long']

print([bedroom_squared.mean(),bed_bath_rooms.mean(),lof_sqft_living.mean(), lat_plus_long.mean()])

[12.4466777015843, 7.5039016315913925, 7.550274679645938, -74.65333355403168]


In [15]:
df['bedroom_squared'] = df['bedrooms'] ** 2
df['bed_bath_rooms'] = df['bedrooms'] * df['bathrooms']
df['lof_sqft_living'] = np.log(df['sqft_living'])
df['lat_plus_long'] = df['lat'] + df['long']

In [16]:
f1 = ['sqft_living', 'bedrooms', 'bathrooms', 'lat','long']
f2 = ['sqft_living', 'bedrooms', 'bathrooms', 'lat','long','bed_bath_rooms']
f3 = ['sqft_living', 'bedrooms', 'bathrooms', 'lat','long', 'bed_bath_rooms', 'bedroom_squared', 'lof_sqft_living', 'lat_plus_long']

In [17]:
df_1 = df[f1]
df_2 = df[f2]
df_3 = df[f3]

In [18]:
lr1 = LinearRegression()
lr1.fit(df_1,df['price'])
lr1.coef_

array([ 3.12258646e+02, -5.95865332e+04,  1.57067421e+04,  6.58619264e+05,
       -3.09374351e+05])

In [19]:
lr2 = LinearRegression()
lr2.fit(df_2,df['price'])
lr2.coef_

array([ 3.06610053e+02, -1.13446368e+05, -7.14613083e+04,  6.54844630e+05,
       -2.94298969e+05,  2.55796520e+04])

In [20]:
lr3 = LinearRegression()
lr3.fit(df_3,df['price'])
lr3.coef_

array([ 5.29422820e+02,  3.45142296e+04,  6.70607813e+04,  5.34085611e+05,
       -4.06750711e+05, -8.57050439e+03, -6.78858667e+03, -5.61831484e+05,
        1.27334900e+05])

In [21]:
def rss(y_pred, y):
    return sum([(y_pred[i]-y[i])**2 for i in range(len(y_pred))])

In [22]:
sum((lr1.predict(df_1) - df['price'])**2)

967879963049546.2

In [23]:
sum((lr2.predict(df_2) - df['price'])**2)

958419635074070.4

In [24]:
sum((lr3.predict(df_3) - df['price'])**2)

903436455050478.4

In [25]:
test_df['bedroom_squared'] = test_df['bedrooms'] ** 2
test_df['bed_bath_rooms'] = test_df['bedrooms'] * df['bathrooms']
test_df['lof_sqft_living'] = np.log(test_df['sqft_living'])
test_df['lat_plus_long'] = test_df['lat'] + df['long']

test_df1 = test_df[f1]
test_df2 = test_df[f2]
test_df3 = test_df[f3]

test_lr1 = LinearRegression()
test_lr1.fit(test_df1,test_df['price'])
test_lr2 = LinearRegression()
test_lr2.fit(test_df2,test_df['price'])
test_lr3 = LinearRegression()
test_lr3.fit(test_df3,test_df['price'])

LinearRegression()

In [26]:
sum((lr1.predict(test_df1) - test_df['price'])**2)

225500469795490.2

In [27]:
sum((lr2.predict(test_df2) - test_df['price'])**2)

273964318595573.0

In [28]:
sum((lr3.predict(test_df3) - test_df['price'])**2)

262069381007136.75

In [29]:
2.5e7

25000000.0

In [30]:
def get_numpy_data(data_sframe, features, output):
    df['constant'] = 1 # add a constant column to an SFrame
    # prepend variable 'constant' to the features list
    features = ['constant'] + features
    # select the columns of data_SFrame given by the ‘features’ list into the SFrame ‘features_sframe’
    features_sframe = df[features]
    # this will convert the features_sframe into a numpy matrix:
    features_matrix = features_sframe.to_numpy()
    # assign the column of data_sframe associated with the target to the variable ‘output_sarray’
    output_sarray = data_sframe[output]
    # this will convert the SArray into a numpy array:
    output_array = output_sarray.to_numpy()
    return(features_matrix, output_array)

In [31]:
def predict_outcome(feature_matrix, weights):
    predictions = feature_matrix @  weights
    return(predictions)

In [32]:
def feature_derivative(errors, feature):
    derivative = 2*np.dot(errors,feature)
    return(derivative)

In [75]:
def regression_gradient_descent(feature_matrix, output, initial_weights, step_size, tolerance):
    converged = False
    weights = np.array(initial_weights)
    while not converged:
        # compute the predictions based on feature_matrix and weights:
        predictions = predict_outcome(feature_matrix,weights)
        # compute the errors as predictions - output:
        error = predictions - output
        gradient_sum_squares = 0 # initialize the gradient
        # while not converged, update each weight individually:
        for i in range(len(weights)):
            # Recall that feature_matrix[:, i] is the feature column associated with weights[i]
            # compute the derivative for weight[i]:
            derivation = feature_derivative(error, feature_matrix[:,i])
            # add the squared derivative to the gradient magnitude
            gradient_sum_squares += derivation ** 2
            # update the weight based on step size and derivative:
            weights[i] -= step_size*derivation
        gradient_magnitude = np.sqrt(gradient_sum_squares)
        if gradient_magnitude < tolerance:
            converged = True
    return(weights)

In [77]:
simple_features = ['sqft_living']
my_output= 'price'
(simple_feature_matrix, output) = get_numpy_data(df, simple_features, my_output)
initial_weights = np.array([-47000., 1.])
step_size = 7e-12
tolerance = 2.5e7

In [83]:
simple_weights = regression_gradient_descent(simple_feature_matrix, output,initial_weights, step_size,tolerance)

In [85]:
simple_weights

array([-46999.88716555,    281.91211918])

In [81]:
print('weight of sqft_living',simple_weights[1])

weight of sqft_living 281.91211917520917


In [109]:
test_df['sqft_living'][0]*simple_weights[1] + simple_weights[0]

356134.4432550024

In [99]:
model_features = ['sqft_living', 'sqft_living15']
my_output = 'price'
(feature_matrix, output) = get_numpy_data(df, model_features,my_output)
initial_weights = np.array([-100000., 1., 1.])
step_size = 4e-12
tolerance = 1e9

In [101]:
simple_weights2 = regression_gradient_descent(feature_matrix, output,initial_weights, step_size,tolerance)

In [115]:
simple_weights2

array([-9.99999688e+04,  2.45072603e+02,  6.52795267e+01])

In [127]:
test_df['sqft_living'][0]*simple_weights2[1] + test_df['sqft_living15'][0]*simple_weights2[2]+simple_weights2[0]

366651.4116294939

In [111]:
test_df['price'][0]

310000.0

In [121]:
RSS1 = sum((np.dot(test_df['sqft_living'],simple_weights[1]) + simple_weights[0] - test_df['price'])**2)
RSS1

275400044902128.3

In [123]:
RSS2 = sum((np.dot(test_df[['sqft_living','sqft_living15']],simple_weights2[1:]) + simple_weights2[0] - test_df['price'])**2)
RSS2

270263443629803.56